In [ ]:
# Construye el modelo
import pyomo.environ as pe

# Resuelve el modelo
import pyomo.opt as po

In [ ]:
model = pe.ConcreteModel()

Sets

In [ ]:
model.product_types = pe.Set(initialize = ["A","B","C"])
model.countries = pe.Set(initialize = ["Spain","France","Germany","Austria","Switzerland","Italy",])

Parameters

In [ ]:
cost_dict = {
    ("A", "Spain"): 160, ("A", "France"): 210, ("A", "Germany"): 180, ("A", "Austria"): 110, ("A", "Switzerland"): 85, ("A", "Italy"): 170,
    ("B", "Spain"): 120, ("B", "France"): 240, ("B", "Germany"): 165, ("B", "Austria"): 135, ("B", "Switzerland"): 100, ("B", "Italy"): 160,
    ("C", "Spain"): 150, ("C", "France"): 200, ("C", "Germany"): 175, ("C", "Austria"): 140, ("C", "Switzerland"): 115, ("C", "Italy"): 135
}

model.unitary_cost = pe.Param(model.product_types, model.factories, initialize=cost_dict)
model.change_cost = pe.Param(initialize=25)

In [ ]:
countries = [
    "Spain",
    "France",
    "Germany",
    "Austria",
    "Switzerland",
    "Italy"
]

# Países desde los que podemos cambiar de coche
model.change_countries = pe.Set(
    initialize=countries[:-1]
)

# País siguiente a cada país
next_country = dict(zip(countries[:-1], countries[1:]))

## Variables de decisión

In [ ]:
model.x = pe.Var(model.countries, model.product_types, within=pe.Binary)

# y[c] = 1 si se cambia de tipo de coche al salir del país c.
# Italia no se incluye: desde allí el turista regresa a España en avión.
model.y = pe.Var(model.change_countries, within=pe.Binary)

## Función objetivo

Minimizar el coste del combustible más 25 € por cada cambio de tipo de coche.

In [ ]:
def obj_rule(model):
    fuel_cost = sum(
        model.unitary_cost[t, c] * model.x[c, t]
        for c in model.factories
        for t in model.product_types
    )
    switching_cost = sum(
        model.change_cost * model.y[c]
        for c in model.change_countries
    )
    return fuel_cost + switching_cost


model.total_cost = pe.Objective(rule=obj_rule, sense=pe.minimize)

## Restricciones

En cada país se elige exactamente un tipo de coche. Las dos desigualdades de cambio linealizan el valor absoluto de los apuntes: $|x_{c,t}-x_{c+1,t}|\le y_c$.

In [ ]:
# Exactamente un tipo de coche en cada país.
def one_car_rule(model, c):
    return sum(model.x[c, t] for t in model.product_types) == 1


model.one_car = pe.Constraint(model.factories, rule=one_car_rule)

In [ ]:
# Si un tipo de coche deja de utilizarse entre dos países, y[c] debe valer 1.
def change_rule_1(model, c, t):
    next_c = next_country[c]
    return model.x[c, t] - model.x[next_c, t] <= model.y[c]


model.change_1 = pe.Constraint(
    model.change_countries, model.product_types, rule=change_rule_1
)

In [ ]:
# Si un tipo de coche empieza a utilizarse entre dos países, y[c] debe valer 1.
def change_rule_2(model, c, t):
    next_c = next_country[c]
    return model.x[next_c, t] - model.x[c, t] <= model.y[c]


model.change_2 = pe.Constraint(
    model.change_countries, model.product_types, rule=change_rule_2
)

## Resolver con Gurobi

Se conserva el solver `gurobi_direct` de tu notebook. Necesita `pyomo`, `gurobipy` y una licencia de Gurobi activa en el entorno de Python que ejecuta el notebook. No hace falta indicar la ruta de `gurobi_cl.exe` a `gurobi_direct`.

In [ ]:
solver = po.SolverFactory("gurobi_direct")

if not solver.available(exception_flag=False):
    raise RuntimeError(
        "Gurobi no está disponible en este kernel. "
        "Comprueba que pyomo y gurobipy estén instalados y que la licencia esté activa."
    )

results = solver.solve(model, tee=True)

print("Estado:", results.solver.status)
print("Terminación:", results.solver.termination_condition)

if results.solver.termination_condition != po.TerminationCondition.optimal:
    raise RuntimeError("Gurobi no ha encontrado una solución óptima; revisa el estado del solver.")

## Solución óptima

In [ ]:
fuel_cost = sum(
    pe.value(model.unitary_cost[t, c]) * pe.value(model.x[c, t])
    for c in model.factories
    for t in model.product_types
)
switching_cost = sum(
    pe.value(model.change_cost) * pe.value(model.y[c])
    for c in model.change_countries
)

print(f"Coste de combustible: {fuel_cost:.2f} €")
print(f"Coste por cambios:   {switching_cost:.2f} €")
print(f"COSTE TOTAL MÍNIMO:  {pe.value(model.total_cost):.2f} €")

print("\nItinerario óptimo:")
for c in countries:
    car = next(t for t in model.product_types if pe.value(model.x[c, t]) > 0.5)
    price = pe.value(model.unitary_cost[car, c])
    print(f"{c:12s} -> coche {car} (combustible: {price:.0f} €)")

print("\nCambios de coche:")
changes = [c for c in model.change_countries if pe.value(model.y[c]) > 0.5]
if changes:
    for c in changes:
        print(f"{c} -> {next_country[c]} (+{pe.value(model.change_cost):.0f} €)")
else:
    print("Ninguno")

assert abs(pe.value(model.total_cost) - fuel_cost - switching_cost) < 1e-6